# Borusyak–Jaravel–Spiess Imputation DiD

**Econometrics Notebook Library · transparent estimator and diagnostic audit**

This notebook has two deliberately separate layers:

1. **Reported estimates and standard errors use existing software.** The maintained Python package [`did-imputation`](https://pypi.org/project/did-imputation/), which is linked from Kirill Borusyak's [code page](https://sites.google.com/view/borusyak/code) and mirrors the authors' Stata command, is the primary implementation.
2. **The point-estimator mechanics are the main pedagogical object.** The three imputation steps are then reconstructed line by line in visible cells and checked numerically against the package.

There is no repository-specific BJS estimation command in this notebook. The manual calculation is an audit of the point estimator, not a substitute for the package's support checks or inference.

## 1. Estimand and identifying restrictions

Treatment is absorbing: unit $i$ is treated from date $E_i$ onward. In the benchmark specification, untreated potential outcomes satisfy

$$
Y_{it}(0)=\alpha_i+\lambda_t+\varepsilon_{it},
$$

and treatment has no anticipatory effect. Treatment effects after adoption may otherwise be heterogeneous across units and time. For treated observations,

$$
\tau_{it}=Y_{it}(1)-Y_{it}(0).
$$

The event-time target used here is the equal-weight average among treated observations observed $h$ periods after adoption:

$$
\tau_h=\frac{1}{N_h}\sum_{i,t:\,t-E_i=h}\tau_{it}.
$$

The estimator first learns $Y(0)$ from untreated observations, then imputes treated counterfactuals, and only then applies the target weights.

## 2. Known-truth staggered-adoption simulation

The simulation has serially correlated errors and heterogeneous, dynamic effects. Untreated outcomes satisfy the additive unit and time fixed-effect model. Never-treated units have a missing treatment date, as expected by the package.

In [ ]:
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from did_imputation import did_imputation, event_plot
from econnotes.panel import simulate_staggered_panel

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

df = simulate_staggered_panel(n_units=260, seed=51)
df["first_treat"] = df["cohort"].replace(np.inf, np.nan)

pd.Series({
    "did-imputation version": version("did-imputation"),
    "observations": len(df),
    "units": df["unit"].nunique(),
    "periods": df["time"].nunique(),
})

In [ ]:
cohort_summary = (
    df[["unit", "cohort"]]
    .drop_duplicates()
    .assign(cohort_label=lambda x: x["cohort"].replace(np.inf, "never"))
    .groupby("cohort_label", dropna=False)
    .size()
    .rename("units")
)
cohort_summary

## 3. Support comes before estimation

With unit and period fixed effects, every treated unit must have untreated history, and every treated calendar period must retain untreated observations. Richer fixed effects or controls create additional support requirements; the package performs the corresponding computational checks.

In [ ]:
untreated = df["treated"].eq(0)
treated = df["treated"].eq(1)

missing_unit_support = sorted(
    set(df.loc[treated, "unit"]) - set(df.loc[untreated, "unit"])
)
missing_time_support = sorted(
    set(df.loc[treated, "time"]) - set(df.loc[untreated, "time"])
)

support_check = pd.Series({
    "treated units without untreated history": len(missing_unit_support),
    "treated periods without untreated observations": len(missing_time_support),
    "untreated observations": int(untreated.sum()),
    "treated observations": int(treated.sum()),
})

assert not missing_unit_support
assert not missing_time_support
support_check

## 4. Primary estimation with the maintained package

The package supplies the production calculation, support handling, cluster-robust conservative standard errors, and the paper's separate pre-trend regression. Five post-treatment horizons are requested; standard errors cluster by unit by default.

In [ ]:
HORIZONS = list(range(5))
K_PRE = 3

package_fit = did_imputation(
    df.copy(),
    "y",
    "unit",
    "time",
    "first_treat",
    horizons=HORIZONS,
    pretrends=K_PRE,
)

package_event = pd.DataFrame({
    "event_time": [int(name.removeprefix("tau")) for name in package_fit.estimates],
    "estimate": [float(value) for value in package_fit.estimates.values()],
    "se": [float(package_fit.std_errors[name]) for name in package_fit.estimates],
})
package_event["ci_low"] = package_event["estimate"] - 1.96 * package_event["se"]
package_event["ci_high"] = package_event["estimate"] + 1.96 * package_event["se"]
package_event

In [ ]:
truth = (
    df.loc[treated]
    .groupby("event_time", as_index=False)["tau_true"]
    .mean()
    .rename(columns={"tau_true": "truth"})
)

reported = package_event.merge(truth, on="event_time", how="left")

fig, ax = plt.subplots()
ax.errorbar(
    reported["event_time"],
    reported["estimate"],
    yerr=1.96 * reported["se"],
    marker="o",
    capsize=3,
    label="did-imputation (95% CI)",
)
ax.plot(reported["event_time"], reported["truth"], marker="s", label="Known truth")
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="Event time", ylabel="Effect", title="Package estimates against the known treatment-effect path")
ax.legend();

# The estimator, made visible

The next three sections are the main pedagogical content. They reproduce the point estimator directly in the notebook. They do **not** attempt to recreate the package's conservative variance estimator; a naive standard deviation of imputed effects would ignore shared first-stage estimation and within-unit dependence.

## 5. Step 1 — fit the untreated outcome model on untreated observations only

Estimate

$$
Y_{it}=\alpha_i+\lambda_t+\varepsilon_{it}
$$

only where $D_{it}=0$. Treated outcomes cannot be used to fit their own counterfactuals. The simulated covariate `x` is time invariant and therefore absorbed by unit fixed effects.

In [ ]:
fit_y0 = smf.ols(
    "y ~ C(unit) + C(time)",
    data=df.loc[untreated],
).fit()

pd.Series({
    "observations used in step 1": int(fit_y0.nobs),
    "treated observations used in step 1": int(df.loc[fit_y0.model.data.row_labels, "treated"].sum()),
    "R-squared on untreated sample": float(fit_y0.rsquared),
})

## 6. Step 2 — impute $\widehat Y_{it}(0)$ and observation-level effects

Extrapolate the fitted fixed effects to treated observations and compute

$$
\widehat\tau_{it}=Y_{it}-\widehat Y_{it}(0).
$$

In [ ]:
work = df.copy()
work["y0_hat"] = fit_y0.predict(work)
work["tau_hat"] = np.where(
    work["treated"].eq(1),
    work["y"] - work["y0_hat"],
    np.nan,
)

work.loc[work["treated"].eq(1), [
    "unit", "time", "cohort", "event_time", "y", "y0_hat", "tau_hat"
]].head(10)

## 7. Step 3 — apply the estimand's weights

For each horizon, take the equal-weight mean of the imputed observation-level effects. Alternative substantive targets require alternative prespecified weights.

In [ ]:
manual_event = (
    work.loc[work["treated"].eq(1) & work["event_time"].isin(HORIZONS)]
    .groupby("event_time", as_index=False)
    .agg(manual_estimate=("tau_hat", "mean"), n=("tau_hat", "size"))
    .sort_values("event_time")
)

comparison = manual_event.merge(
    package_event[["event_time", "estimate"]].rename(columns={"estimate": "package_estimate"}),
    on="event_time",
    how="left",
)
comparison["absolute_difference"] = (
    comparison["manual_estimate"] - comparison["package_estimate"]
).abs()

assert comparison["absolute_difference"].max() < 1e-6
comparison

# Diagnostics are separate from effect estimation

Section 4.4 of BJS proposes a **family** of robust OLS tests. Choose an observable violation model $W_{it}$, estimate on untreated observations only,

$$
Y_{it}=\alpha_i+\lambda_t+W_{it}'\gamma+u_{it},
$$

and test $H_0:\gamma=0$ using a heteroskedasticity- and cluster-robust Wald test. The choice of $W_{it}$ should reflect a prespecified scientific concern. A lead-indicator plot is one choice, not the definition of the entire diagnostic strategy.

## 8. BJS option A — near-event lead indicators

Here, $W_{it}$ contains indicators for the three periods immediately before adoption. Earlier pre-treatment observations and never-treated observations form the reference group. This choice primarily probes anticipation or event-time-local departures.

In [ ]:
diag = df.copy()
for j in range(1, K_PRE + 1):
    diag[f"pre{j}"] = (
        diag["event_time"].eq(-j) & diag["treated"].eq(0)
    ).astype(int)

pre_terms = [f"pre{j}" for j in range(1, K_PRE + 1)]
pre_fit = smf.ols(
    "y ~ C(unit) + C(time) + " + " + ".join(pre_terms),
    data=diag.loc[diag["treated"].eq(0)],
).fit(
    cov_type="cluster",
    cov_kwds={"groups": diag.loc[diag["treated"].eq(0), "unit"]},
)

joint_pre = pre_fit.wald_test(
    ", ".join(f"{term} = 0" for term in pre_terms),
    use_f=True,
    scalar=True,
)

pretrend_table = pd.DataFrame({
    "event_time": [-j for j in range(1, K_PRE + 1)],
    "estimate": [float(pre_fit.params[f"pre{j}"]) for j in range(1, K_PRE + 1)],
    "cluster_se": [float(pre_fit.bse[f"pre{j}"]) for j in range(1, K_PRE + 1)],
    "package_estimate": [
        float(package_fit.pretrends_estimates[f"pre{j}"])
        for j in range(1, K_PRE + 1)
    ],
}).sort_values("event_time")

assert np.max(np.abs(pretrend_table["estimate"] - pretrend_table["package_estimate"])) < 1e-6

pretrend_table, {
    "joint F statistic": float(joint_pre.statistic),
    "joint p-value": float(joint_pre.pvalue),
}

In [ ]:
event_plot(
    results_obj=package_fit,
    together=False,
    title="BJS event-study display: pre-trend test and effects estimated separately",
    xlabel="Event time",
    ylabel="Coefficient",
    capsize=3,
);

## 9. BJS option B — a structural-break violation model

BJS explicitly note that $W_{it}$ can target parallel-trends violations rather than only anticipation—for example, a structural break in unit effects. To keep this simulation low dimensional, use a grouped version: cohort-specific intercept shifts after a prespecified calendar breakpoint, with never-treated units as the reference. In an application, the breakpoint and grouping must come from institutional knowledge rather than outcome mining.

In [ ]:
calendar_break = 2
break_diag = df.copy()
treated_cohorts = sorted(int(g) for g in break_diag.loc[np.isfinite(break_diag["cohort"]), "cohort"].unique())
break_terms = []

for cohort in treated_cohorts:
    name = f"break_g{cohort}"
    break_diag[name] = (
        (break_diag["time"] >= calendar_break)
        & break_diag["cohort"].eq(cohort)
    ).astype(int)
    break_terms.append(name)

break_sample = break_diag.loc[break_diag["treated"].eq(0)]
break_fit = smf.ols(
    "y ~ C(unit) + C(time) + " + " + ".join(break_terms),
    data=break_sample,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": break_sample["unit"]},
)

joint_break = break_fit.wald_test(
    ", ".join(f"{term} = 0" for term in break_terms),
    use_f=True,
    scalar=True,
)

break_table = pd.DataFrame({
    "cohort": treated_cohorts,
    "break_estimate": [float(break_fit.params[name]) for name in break_terms],
    "cluster_se": [float(break_fit.bse[name]) for name in break_terms],
})

break_table, {
    "breakpoint": calendar_break,
    "joint F statistic": float(joint_break.statistic),
    "joint p-value": float(joint_break.pvalue),
}

## 10. Liu–Wang–Xu placebo — related, but not the BJS test

Liu, Wang and Xu (published online in 2022; AJPS 2024) propose hiding several periods immediately before treatment, predicting them out of sample, and treating their average prediction errors as placebo effects. BJS contrast this placebo strategy with their robust OLS test.

With $S=3$, shift the nominal treatment date three periods earlier and estimate pseudo-effects only for shifted horizons 0, 1 and 2—actual event times $-3,-2,-1$. This uses the same package estimator, not residuals from the full untreated-sample fit.

In [ ]:
S_PLACEBO = 3
placebo_df = df.copy()
placebo_df["w_placebo"] = (
    placebo_df["event_time"].between(-S_PLACEBO, -1)
    & np.isfinite(placebo_df["cohort"])
).astype(float)

placebo_path_fit = did_imputation(
    placebo_df.copy(),
    "y",
    "unit",
    "time",
    "first_treat",
    shift=S_PLACEBO,
    horizons=list(range(S_PLACEBO)),
    minn=0,
)
placebo_average_fit = did_imputation(
    placebo_df.copy(),
    "y",
    "unit",
    "time",
    "first_treat",
    shift=S_PLACEBO,
    wtr=["w_placebo"],
    minn=0,
)

placebo_path = pd.DataFrame({
    "actual_event_time": [
        int(name.removeprefix("tau")) - S_PLACEBO
        for name in placebo_path_fit.estimates
    ],
    "estimate": [float(value) for value in placebo_path_fit.estimates.values()],
    "se": [
        float(placebo_path_fit.std_errors[name])
        for name in placebo_path_fit.estimates
    ],
}).sort_values("actual_event_time")

placebo_average = float(placebo_average_fit.estimates["w_placebo"])
placebo_average_se = float(placebo_average_fit.std_errors["w_placebo"])
placebo_z = placebo_average / placebo_average_se
placebo_p = 2 * stats.norm.sf(abs(placebo_z))

placebo_path, {
    "average placebo effect": placebo_average,
    "standard error": placebo_average_se,
    "two-sided p-value": float(placebo_p),
}

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    placebo_path["actual_event_time"],
    placebo_path["estimate"],
    yerr=1.96 * placebo_path["se"],
    marker="o",
    capsize=3,
)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xlabel="Actual event time",
    ylabel="Out-of-sample placebo effect",
    title=f"Liu–Wang–Xu-style placebo window; average p = {placebo_p:.3f}",
);

The placebo null test can be underpowered. Liu–Wang–Xu also discuss equivalence testing, which requires a substantively prespecified equivalence range. No such margin is invented for this generic simulation. The BJS lead test, BJS structural-break test, and Liu–Wang–Xu placebo answer related but non-identical questions.

# A failure case: cohort-specific untreated trends

Passing diagnostics in one simulated design is not evidence that imputation is automatically valid. Add cohort-specific linear drift to untreated potential outcomes while leaving the true treatment effects unchanged. This violates the maintained common-trend model.

In [ ]:
violated = df.copy()
cohort_slopes = violated["cohort"].map({4.0: 0.25, 6.0: 0.15, 8.0: 0.05}).fillna(0.0)
untreated_drift = cohort_slopes * violated["time"]
violated["y0"] = violated["y0"] + untreated_drift
violated["y"] = violated["y"] + untreated_drift

violated_package = did_imputation(
    violated.copy(),
    "y",
    "unit",
    "time",
    "first_treat",
    horizons=HORIZONS,
    minn=0,
    nose=True,
)

violated_event = pd.DataFrame({
    "event_time": [int(name.removeprefix("tau")) for name in violated_package.estimates],
    "estimate_under_violation": [float(value) for value in violated_package.estimates.values()],
}).merge(truth, on="event_time", how="left")
violated_event["bias"] = violated_event["estimate_under_violation"] - violated_event["truth"]
violated_event

In [ ]:
violated_diag = violated.copy()
violated_untreated = violated_diag["treated"].eq(0)
for j in range(1, K_PRE + 1):
    violated_diag[f"pre{j}"] = (
        violated_diag["event_time"].eq(-j) & violated_untreated
    ).astype(int)

violated_pre_fit = smf.ols(
    "y ~ C(unit) + C(time) + " + " + ".join(pre_terms),
    data=violated_diag.loc[violated_untreated],
).fit(
    cov_type="cluster",
    cov_kwds={"groups": violated_diag.loc[violated_untreated, "unit"]},
)
violated_joint = violated_pre_fit.wald_test(
    ", ".join(f"{term} = 0" for term in pre_terms),
    use_f=True,
    scalar=True,
)

assert violated_event["bias"].abs().max() > 0.25
assert float(violated_joint.pvalue) < 0.01

{
    "largest absolute bias": float(violated_event["bias"].abs().max()),
    "BJS lead-test F statistic": float(violated_joint.statistic),
    "BJS lead-test p-value": float(violated_joint.pvalue),
}

## Researcher checklist

- Use maintained software for reported BJS estimates and inference.
- Put the three point-estimator steps in visible cells when the goal is pedagogical transparency.
- Audit imputation support before interpreting an ATT.
- State the target observations, horizons, and weights before aggregation.
- Keep pre-trend testing separate from post-treatment estimation.
- In the BJS test, choose $W_{it}$ to represent a prespecified plausible violation; lead indicators are only one option.
- Label the Liu–Wang–Xu out-of-sample placebo as a distinct diagnostic, not as the BJS test.
- Treat failure to reject as weak evidence, not proof of parallel trends.
- Demonstrate at least one violation under which imputation is biased.

### References and implementations

- Borusyak, K., Jaravel, X., & Spiess, J. (2024). *Revisiting Event-Study Designs: Robust and Efficient Estimation*. **Review of Economic Studies, 91**(6), 3253–3285. https://doi.org/10.1093/restud/rdae007
- BJS Section 4.4 and the authors' Stata implementation: https://github.com/borusyak/did_imputation
- Python implementation used here (`did-imputation` 0.1.0), authored by Georgii Marinichev and linked from Borusyak's code page: https://pypi.org/project/did-imputation/
- Liu, L., Wang, Y., & Xu, Y. (2024; published online 2022). *A Practical Guide to Counterfactual Estimators for Causal Inference with Time-Series Cross-Sectional Data*. **American Journal of Political Science, 68**(1), 160–176. https://doi.org/10.1111/ajps.12723